#   Recharger le fichier original

In [3]:
import json

with open("../data/export.events.clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Fichier original rechargé")
print("Nombre d'events :", len(data))

Fichier original rechargé
Nombre d'events : 11948


# premier code modifié, avec cache et infos utiles pour le 2e code

In [4]:
import requests
import pandas as pd
from IPython.display import display, Markdown

headers = {
    "User-Agent": "MyResearchNotebook/1.0"
}

entity_cache = {}

def get_qid_from_uri(uri):
    if not uri or "wikidata.org/entity/" not in str(uri):
        return None
    return str(uri).rstrip("/").split("/")[-1]

def get_label(entity):
    labels = entity.get("labels", {})
    if "fr" in labels:
        return labels["fr"]["value"]
    if "en" in labels:
        return labels["en"]["value"]
    return None

def get_description(entity):
    descriptions = entity.get("descriptions", {})
    if "fr" in descriptions:
        return descriptions["fr"]["value"]
    if "en" in descriptions:
        return descriptions["en"]["value"]
    return None

def fetch_entity(qid):
    if qid in entity_cache:
        return entity_cache[qid]

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
    response = requests.get(url, timeout=20, headers=headers)
    response.raise_for_status()
    entity = response.json()["entities"].get(qid)
    entity_cache[qid] = entity
    return entity

# 1. Récupérer tous les QID uniques des Place
place_qids = set()

for event in data:
    for node in event.get("nodes", []):
        if "Thing/Abstract/Location/Place" in node.get("labels", []):
            qid = get_qid_from_uri(node.get("properties", {}).get("uri"))
            if qid:
                place_qids.add(qid)

print("Nombre total de QID Place uniques :", len(place_qids))

# 2. Construire les dictionnaires utiles
qid_to_country = {}
qid_to_status = {}
qid_to_country_qid = {}
qid_to_country_entity = {}

qids_list = list(place_qids)
total = len(qids_list)

for i, qid in enumerate(qids_list, start=1):
    if i == 1 or i % 10 == 0 or i == total:
        print(f"Progression Wikidata : {i}/{total}")

    try:
        entity = fetch_entity(qid)
        if entity is None:
            qid_to_country[qid] = None
            qid_to_status[qid] = "entity_not_found"
            qid_to_country_qid[qid] = None
            qid_to_country_entity[qid] = None
            continue

        description = get_description(entity)
        description_low = description.lower() if description else ""

        if "country" in description_low or "pays" in description_low or description_low.startswith("état") or description_low.startswith("etat"):
            qid_to_country[qid] = get_label(entity)
            qid_to_status[qid] = "place_is_already_country"
            qid_to_country_qid[qid] = qid
            qid_to_country_entity[qid] = entity
            continue

        claims = entity.get("claims", {})
        if "P17" not in claims:
            qid_to_country[qid] = None
            qid_to_status[qid] = "no_country_relation"
            qid_to_country_qid[qid] = None
            qid_to_country_entity[qid] = None
            continue

        country_qid = claims["P17"][0]["mainsnak"]["datavalue"]["value"]["id"]
        country_entity = fetch_entity(country_qid)

        qid_to_country[qid] = get_label(country_entity)
        qid_to_status[qid] = "country_found_via_p17"
        qid_to_country_qid[qid] = country_qid
        qid_to_country_entity[qid] = country_entity

    except Exception as e:
        qid_to_country[qid] = None
        qid_to_status[qid] = f"wikidata_fetch_failed: {repr(e)}"
        qid_to_country_qid[qid] = None
        qid_to_country_entity[qid] = None

print("Récupération Wikidata terminée.")

# 3. Tableau de contrôle
rows = []
total_events = len(data)

for idx, event in enumerate(data, start=1):
    if idx == 1 or idx % 200 == 0 or idx == total_events:
        print(f"Progression JSON : {idx}/{total_events}")

    rid = event.get("resultAnalyseId")
    event_type = event.get("type", "UNKNOWN")
    context = event.get("context", "")

    for node in event.get("nodes", []):
        if "Thing/Abstract/Location/Place" not in node.get("labels", []):
            continue

        form = str(node.get("form", "")).strip()
        qid = get_qid_from_uri(node.get("properties", {}).get("uri"))

        if qid:
            country_to_add = qid_to_country.get(qid)
            status = qid_to_status.get(qid, "unknown")
            country_qid = qid_to_country_qid.get(qid)
        else:
            country_to_add = None
            status = "no_wikidata_uri"
            country_qid = None

        rows.append({
            "resultAnalyseId": rid,
            "event_type": event_type,
            "place_form": form,
            "qid": qid,
            "country_to_add": country_to_add,
            "country_qid": country_qid,
            "status": status,
            "context": context if len(str(context)) <= 120 else str(context)[:120] + "..."
        })

df_place_country_all = pd.DataFrame(rows)

display(Markdown("## Enrichissement de tous les nœuds `Place`"))
display(df_place_country_all)

summary_df = pd.DataFrame({
    "Indicateur": [
        "Nombre total de QID Place uniques",
        "Nombre de lignes produites",
        "Pays trouvés via Wikidata",
        "Cas déjà pays",
        "Cas sans pays trouvé",
        "Entités Wikidata en cache"
    ],
    "Valeur": [
        len(place_qids),
        len(df_place_country_all),
        (df_place_country_all["status"] == "country_found_via_p17").sum(),
        (df_place_country_all["status"] == "place_is_already_country").sum(),
        df_place_country_all["country_to_add"].isna().sum(),
        len(entity_cache)
    ]
})

display(Markdown("## Résumé"))
display(summary_df.style.hide(axis="index"))

Nombre total de QID Place uniques : 240
Progression Wikidata : 1/240
Progression Wikidata : 10/240
Progression Wikidata : 20/240
Progression Wikidata : 30/240
Progression Wikidata : 40/240
Progression Wikidata : 50/240
Progression Wikidata : 60/240
Progression Wikidata : 70/240
Progression Wikidata : 80/240
Progression Wikidata : 90/240
Progression Wikidata : 100/240
Progression Wikidata : 110/240
Progression Wikidata : 120/240
Progression Wikidata : 130/240
Progression Wikidata : 140/240
Progression Wikidata : 150/240
Progression Wikidata : 160/240
Progression Wikidata : 170/240
Progression Wikidata : 180/240
Progression Wikidata : 190/240
Progression Wikidata : 200/240
Progression Wikidata : 210/240
Progression Wikidata : 220/240
Progression Wikidata : 230/240
Progression Wikidata : 240/240
Récupération Wikidata terminée.
Progression JSON : 1/11948
Progression JSON : 200/11948
Progression JSON : 400/11948
Progression JSON : 600/11948
Progression JSON : 800/11948
Progression JSON : 10

## Enrichissement de tous les nœuds `Place`

,resultAnalyseId,event_type,place_form,qid,country_to_add,country_qid,status,context
0,698b35e042a8083a290c11c6,Thing/Abstract/Event/Addition,région,None,None,None,no_wikidata_uri,"sans électricité dans l’oblast de Zaporijia, s..."
1,698b35e042a8083a290c11cb,Thing/Abstract/Event/Put,Cuba,Q241,Cuba,Q241,place_is_already_country,La compagnie aérienne Air Canada a suspendu lu...
2,698b35e042a8083a290c11ca,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Atlantique,Q97,None,None,wikidata_fetch_failed: KeyError('datavalue'),"bricant finlandais, dont les antennes, essenti..."
3,698b35e942a8083a290c11ce,Thing/Abstract/Event/Killing,Marseille,Q23482,France,Q142,country_found_via_p17,Le procès d'un adolescent de 15 ans pour le me...
4,698b35e742a8083a290c11cd,Thing/Abstract/Event/Attend,Dunkerque,Q45797,France,Q142,country_found_via_p17,"Le chef de l’Etat, Emmanuel Macron, a assisté,..."
...,...,...,...,...,...,...,...,...
993,699e88b74469fb97fca64fbd,Thing/Abstract/Event/Give,Moscou,Q649,duché de Moscou,Q11750528,country_found_via_p17,"z en Chine. La rencontre, hier, à l’Elysée, en..."
994,699e88b74469fb97fca64fbd,Thing/Abstract/Event/Turn,New York,None,None,None,no_wikidata_uri,es présumés de l’assassinat de la militante Ma...
995,699ea4d94469fb97fca64fcc,Thing/Abstract/Event/Gathering/NonDiplomaticGa...,Rio,Q8678,Brésil,Q155,country_found_via_p17,"Lors du carnaval de Rio, au Brésil, mi-février..."
996,699ea4d94469fb97fca64fcc,Thing/Abstract/Event/Give,Brésil,Q155,Brésil,Q155,place_is_already_country,"Lors du carnaval de Rio, au Brésil, mi-février..."


## Résumé

Indicateur,Valeur
Nombre total de QID Place uniques,240
Nombre de lignes produites,998
Pays trouvés via Wikidata,397
Cas déjà pays,264
Cas sans pays trouvé,337
Entités Wikidata en cache,259


# Deuxième code, pour ajouter les nœuds Country dans le JSON, sans retoucher les Place et sans doublons

In [ ]:
import json
import copy

def get_aliases(entity):
    aliases = entity.get("aliases", {})
    if "fr" in aliases:
        return [a["value"] for a in aliases["fr"]]
    if "en" in aliases:
        return [a["value"] for a in aliases["en"]]
    return []

def get_image_url(entity):
    claims = entity.get("claims", {})
    if "P18" not in claims:
        return None
    try:
        image_name = claims["P18"][0]["mainsnak"]["datavalue"]["value"]
        image_name = image_name.replace(" ", "_")
        return f"http://commons.wikimedia.org/wiki/Special:FilePath/{image_name}"
    except Exception:
        return None

def get_coordinates(entity):
    claims = entity.get("claims", {})
    if "P625" not in claims:
        return None, None
    try:
        value = claims["P625"][0]["mainsnak"]["datavalue"]["value"]
        return value.get("latitude"), value.get("longitude")
    except Exception:
        return None, None

def build_country_node_from_place(place_node):
    country_node = copy.deepcopy(place_node)
    country_node["labels"] = ["Thing/Abstract/Location/Country"]
    country_node["properties"]["type"] = "Thing/Abstract/Location/Country"
    return country_node

def build_country_node_from_entity(country_qid, country_entity):
    country_name = get_label(country_entity)
    country_description = get_description(country_entity)
    country_aliases = get_aliases(country_entity)
    country_image = get_image_url(country_entity)
    lat, lon = get_coordinates(country_entity)

    return {
        "_id": f"https://www.wikidata.org/entity/{country_qid}",
        "form": country_name,
        "labels": [
            "Thing/Abstract/Location/Country"
        ],
        "properties": {
            "value": country_name,
            "latitude": lat,
            "longitude": lon,
            "description": country_description,
            "imageUrl": country_image,
            "aliases": country_aliases,
            "type": "Thing/Abstract/Location/Country",
            "info": None,
            "uri": f"https://www.wikidata.org/entity/{country_qid}"
        }
    }

added_count = 0
total_events = len(data)

for idx, event in enumerate(data, start=1):
    if idx == 1 or idx % 200 == 0 or idx == total_events:
        print(f"Progression ajout JSON : {idx}/{total_events}")

    existing_country_keys = {
        (
            str(node.get("_id")),
            tuple(node.get("labels", []))
        )
        for node in event.get("nodes", [])
        if "Thing/Abstract/Location/Country" in node.get("labels", [])
    }

    new_nodes = []

    for node in event.get("nodes", []):
        new_nodes.append(node)

        if "Thing/Abstract/Location/Place" not in node.get("labels", []):
            continue

        qid = get_qid_from_uri(node.get("properties", {}).get("uri"))
        if not qid:
            continue

        status = qid_to_status.get(qid)
        country_name = qid_to_country.get(qid)

        if not country_name:
            continue

        country_node = None

        if status == "place_is_already_country":
            country_node = build_country_node_from_place(node)

        elif status == "country_found_via_p17":
            country_qid = qid_to_country_qid.get(qid)
            country_entity = qid_to_country_entity.get(qid)

            if country_qid and country_entity:
                country_node = build_country_node_from_entity(country_qid, country_entity)

        if country_node is None:
            continue

        country_key = (
            str(country_node.get("_id")),
            tuple(country_node.get("labels", []))
        )

        if country_key not in existing_country_keys:
            new_nodes.append(country_node)
            existing_country_keys.add(country_key)
            added_count += 1

    event["nodes"] = new_nodes

print("Nombre total de nœuds Country ajoutés :", added_count)

output_file = "../data/export.events.with_country_nodes.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Fichier créé :", output_file)

Progression ajout JSON : 1/11948
Progression ajout JSON : 200/11948
Progression ajout JSON : 400/11948
Progression ajout JSON : 600/11948
Progression ajout JSON : 800/11948
Progression ajout JSON : 1000/11948
Progression ajout JSON : 1200/11948
Progression ajout JSON : 1400/11948
Progression ajout JSON : 1600/11948
Progression ajout JSON : 1800/11948
Progression ajout JSON : 2000/11948
Progression ajout JSON : 2200/11948
Progression ajout JSON : 2400/11948
Progression ajout JSON : 2600/11948
Progression ajout JSON : 2800/11948
Progression ajout JSON : 3000/11948
Progression ajout JSON : 3200/11948
Progression ajout JSON : 3400/11948
Progression ajout JSON : 3600/11948
Progression ajout JSON : 3800/11948
Progression ajout JSON : 4000/11948
Progression ajout JSON : 4200/11948
Progression ajout JSON : 4400/11948
Progression ajout JSON : 4600/11948
Progression ajout JSON : 4800/11948
Progression ajout JSON : 5000/11948
Progression ajout JSON : 5200/11948
Progression ajout JSON : 5400/11948